In [ ]:
# Cell 1: load historical log output + coordinated sweep progress.
from pathlib import Path
from collections import deque, Counter
import json, time

# ============================ EDIT THIS PER MACHINE ============================
VM_NAME = "vmA"          # <-- which VM's log to follow (Pod_1 -> VM1, etc.)
# =============================================================================

REPO = Path('/workspace/stable-query-latent')
LOG = Path('/workspace/stable_query_latent_logs') / f'pipeline_{VM_NAME}.log'   # per-VM
OUT_DIR = REPO / 'VICReg_review/heads/cloud_full_sweep_a100'                    # SHARED
EMBED_MANIFEST = REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json'
TEXT_MANIFEST = REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json'


def tail(path=LOG, lines=200):
    path = Path(path)
    print(f'log: {path}')
    if not path.exists():
        print('missing log file (nothing written yet for this VM)')
        return
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in deque(f, maxlen=lines):
            print(line, end='')


def show_manifest(path):
    path = Path(path)
    print(f'\n=== {path.name} ===')
    if not path.exists():
        print('missing')
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('bad json:', exc)
        return
    for key in ['status', 'updated_at', 'finished_at', 'error']:
        if key in data:
            print(f'{key}: {data[key]}')


def show_coordination(out_dir=OUT_DIR):
    # Global progress from the SHARED coordination markers (checkpoint/done.json =
    # done; failed.json = failed; status.json = in flight). Every VM's work shows up
    # here since they share this out_dir. (The ledger is now machine-local scratch.)
    out_dir = Path(out_dir)
    print(f'\n=== coordinated sweep @ {out_dir.name} ===')
    if not out_dir.exists():
        print('out_dir not created yet')
        return
    done = failed = inflight = 0
    inflight_by = Counter()
    for d in out_dir.iterdir():
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        if (d / 'vicreg_review_h5_latest.pt').exists() or (d / 'done.json').exists():
            done += 1
        elif (d / 'failed.json').exists():
            failed += 1
        elif (d / 'status.json').exists():
            inflight += 1
            try:
                inflight_by[json.loads((d / 'status.json').read_text()).get('vm', '?')] += 1
            except Exception:
                pass
    print(f'done={done}  failed={failed}  in-flight={inflight}   in-flight by: {dict(inflight_by)}')
    vmp = out_dir / 'VM_parallel'
    for f in sorted(vmp.glob('*.json')) if vmp.exists() else []:
        try:
            rec = json.loads(f.read_text())
        except Exception:
            rec = {}
        exp = rec.get('expiry')
        live = '' if exp is None else ('(alive)' if exp > time.time() else '(lease expired)')
        print(f"  vm {rec.get('vm', f.stem):16} {live}  {rec.get('info', {})}")


tail(lines=200)
show_manifest(TEXT_MANIFEST)
show_manifest(EMBED_MANIFEST)
show_coordination()
HISTORY_END = LOG.stat().st_size if LOG.exists() else 0

In [ ]:
# Cell 2: start realtime latest log output in the background.
# Re-run this cell to restart the log watcher. It does not stop the training job.
import threading, time
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


START_OFFSET = globals().get('HISTORY_END', None)
WATCHERS = globals().setdefault('WATCHERS', {})


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def read_new_text(path, start_offset):
    if not path.exists():
        return start_offset, ''
    size = path.stat().st_size
    if start_offset is None or start_offset > size:
        start_offset = size
    with path.open('rb') as f:
        f.seek(start_offset)
        data = f.read()
        end_offset = f.tell()
    text = data.decode('utf-8', errors='replace')
    return end_offset, text


def follow(output, stop_event, path=LOG, interval=1, start_offset=START_OFFSET):
    path = Path(path)
    last_offset = start_offset
    last_update = None
    with output:
        print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {path}')
        print('-' * 100)
        print('waiting for new log lines after historical output')
    while not stop_event.is_set():
        if path.exists():
            last_offset, new_text = read_new_text(path, last_offset)
            if new_text:
                last_update = time.strftime('%Y-%m-%d %H:%M:%S')
                with output:
                    print('-' * 100)
                    print(f'new output received at {last_update}')
                    print('-' * 100)
                    print(new_text, end='' if new_text.endswith('\n') else '\n')
        else:
            if last_update is None:
                with output:
                    print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} missing log file: {path}')
        stop_event.wait(interval)


stop_watcher('log')
log_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '520px', 'overflow_y': 'auto'})
display(log_output)
log_stop = threading.Event()
log_thread = threading.Thread(target=follow, args=(log_output, log_stop), daemon=True)
WATCHERS['log'] = {'stop': log_stop, 'thread': log_thread, 'output': log_output}
log_thread.start()
print('log watcher started in background')


In [ ]:
# Cell 3: realtime dashboard in the background (system + sweep progress).
# Renders ONE fixed-size character-art panel that refreshes IN PLACE by
# REPLACING an HTML widget's value each tick -- print+clear_output into an
# Output widget is unreliable from a background thread (the clear message can
# miss the widget context, so lines pile up instead). Pure ASCII on purpose:
# box-drawing/block glyphs (│█░) are ambiguous-width and misalign whenever the
# browser falls back to a CJK font; '|#-.' stay monospace in every font.
# Re-run this cell to restart the watcher; it does not stop the training job.
import html, json, subprocess, threading, time
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

try:
    import psutil
except ImportError:
    psutil = None
    print('psutil is missing. Run: pip install psutil')

WATCHERS = globals().setdefault('WATCHERS', {})
W = 76                    # dashboard inner width (chars)
TICK_SECONDS = 1          # system stats refresh (GPU/CPU/RAM/disk -- local, cheap)
SWEEP_SCAN_SECONDS = 30   # shared-FS sweep rescan (dir walk across MooseFS is much
                          # slower/heavier than psutil; kept slower on purpose so
                          # the dashboard doesn't hammer the network FS every tick)


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == 'max':
            return None
        return int(text)
    except Exception:
        return None


def _f(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def get_memory_status():
    limit = _read_int('/sys/fs/cgroup/memory.max')
    used = _read_int('/sys/fs/cgroup/memory.current')
    if limit is None or used is None:
        limit = _read_int('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        used = _read_int('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, 'cgroup'
    if psutil is None:
        return None, None, None, 'unavailable'
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, 'host'


def get_gpu_stats():
    """Structured per-GPU stats for the dashboard; (rows, error_text).
    nvidia-smi reports memory in MiB (nounits) -> convert to GiB here."""
    try:
        out = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True, text=True, timeout=3,
        ).stdout.strip()
        if not out:
            return [], 'no nvidia-smi output'
        gpus = []
        for line in out.splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 7:
                mem_used, mem_total = _f(parts[3]), _f(parts[4])
                name = parts[1]
                if name.startswith('NVIDIA'):
                    name = name[len('NVIDIA'):].strip()
                gpus.append({
                    'index': parts[0], 'name': name, 'util': _f(parts[2]),
                    'mem_used': mem_used / 1024 if mem_used is not None else None,
                    'mem_total': mem_total / 1024 if mem_total is not None else None,
                    'temp': _f(parts[5]), 'power': _f(parts[6]),
                })
        return gpus, None
    except Exception as exc:
        return [], str(exc)


# Grid size for the sweep progress denominator (best effort; falls back to the
# number of combo dirs seen so far if the sweep config can't be loaded).
try:
    import sys
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    from VICReg_review.sweep.config import SweepConfig
    GRID_TOTAL = len(list(SweepConfig.load(str(REPO / 'VICReg_review/sweep/sweep.yaml')).iter_combos()))
except Exception:
    GRID_TOTAL = None


def scan_sweep(out_dir=OUT_DIR):
    """Light version of check_paralle's buckets, from marker files only. Claims
    are split into live (owner VM lease fresh) vs stale (owner dead/expired --
    reclaimable), so a crashed VM's leftovers don't inflate the live count."""
    out_dir = Path(out_dir)
    if not out_dir.exists():
        return None
    now = time.time()
    stats = {'done': 0, 'failed': 0, 'live': 0, 'stale': 0, 'ckpt': 0,
             'live_by': {}, 'stale_by': {}, 'vms_alive': [], 'vms_total': 0,
             'my_lanes': {}}   # gpu lane -> (combo_id, claim_ts) for THIS machine
    alive = set()
    vm_pids = {}     # fresh name -> current incarnation pid (generation check)
    my_names = set()
    try:
        import socket as _socket
        _host = _socket.gethostname()
    except Exception:
        _host = None
    vmp = out_dir / 'VM_parallel'
    for f in (sorted(vmp.glob('*.json')) if vmp.exists() else []):
        stats['vms_total'] += 1
        try:
            rec = json.loads(f.read_text(encoding='utf-8'))
        except Exception:
            continue
        exp = rec.get('expiry')
        if exp is not None and exp > now:
            name = rec.get('vm', f.stem)
            stats['vms_alive'].append(name)
            alive.add(name)
            vm_pids[name] = rec.get('pid')
            info_host = (rec.get('info') or {}).get('host')
            if (info_host and info_host == _host) or name == VM_NAME \
                    or name.startswith(VM_NAME + '_'):
                my_names.add(name)
    for d in out_dir.iterdir():
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        is_done = (d / 'done.json').exists()
        if not is_done and (d / 'vicreg_review_h5_latest.pt').exists():
            try:
                payload = json.loads((d / 'vicreg_review_h5_manifest.json').read_text(encoding='utf-8'))
                is_done = payload.get('status') == 'done'
            except Exception:
                pass
        if is_done:
            stats['done'] += 1
        elif (d / 'failed.json').exists():
            stats['failed'] += 1
        elif (d / 'status.json').exists():
            try:
                st = json.loads((d / 'status.json').read_text(encoding='utf-8'))
            except Exception:
                st = {}
            vm = st.get('vm', '?')
            # Incarnation check: a claim written by a DEAD same-name
            # predecessor (pid differs from the registry's current pid) is
            # stale, even though the name's lease is fresh -- without this a
            # 50h-old zombie claim shows up as a lane's current combo.
            live = vm in alive
            if live:
                cpid, rpid = st.get('pid'), vm_pids.get(vm)
                if cpid is not None and rpid is not None:
                    try:
                        live = int(cpid) == int(rpid)
                    except (TypeError, ValueError):
                        pass
            kind = 'live' if live else 'stale'
            if kind == 'live' and vm in my_names:
                stats['my_lanes'][st.get('lane')] = (d.name, st.get('ts'))
            stats[kind] += 1
            by = stats[kind + '_by']
            by[vm] = by.get(vm, 0) + 1
        elif (d / 'vicreg_review_h5_latest.pt').exists():
            stats['ckpt'] += 1
    return stats


# ---- character-art rendering (ASCII only, see header comment) ---------------

def bar(frac, width=24):
    frac = 0.0 if frac is None else min(max(float(frac), 0.0), 1.0)
    filled = int(round(frac * width))
    return '#' * filled + '.' * (width - filled)


def box_top(title, stamp):
    head = f'- {title} '
    tail = f' {stamp} -'
    return '+' + head + '-' * max(0, W - len(head) - len(tail)) + tail + '+'


def box_sep(title=''):
    head = f'- {title} ' if title else ''
    return '+' + head + '-' * max(0, W - len(head)) + '+'


def box_row(text=''):
    return '| ' + text[:W - 2].ljust(W - 2) + ' |'


def box_wrap(prefix, text):
    """box_row, but wrap overflow onto continuation rows (at spaces) instead of
    truncating."""
    import textwrap
    width = max(1, W - 2 - len(prefix))
    chunks = textwrap.wrap(text, width, break_long_words=True,
                           break_on_hyphens=False) or ['']
    pad = ' ' * len(prefix)
    return [box_row((prefix if i == 0 else pad) + chunk) for i, chunk in enumerate(chunks)]


def box_bottom():
    return '+' + '-' * W + '+'


def render(gpus, gpu_err, cpu, ram, disk_rw, sweep, sweep_stamp):
    lines = [box_top(f'{VM_NAME} monitor', time.strftime('%Y-%m-%d %H:%M:%S'))]

    if gpu_err:
        lines.append(box_row(f'GPU    n/a ({gpu_err})'))
    for g in gpus:
        util = g['util']
        vram_frac = (g['mem_used'] / g['mem_total']) if g['mem_used'] is not None and g['mem_total'] else None
        temp = f"{g['temp']:.0f}C" if g['temp'] is not None else '?C'
        power = f"{g['power']:.0f}W" if g['power'] is not None else '?W'
        lines.append(box_row(f"GPU{g['index']}  {g['name']}"))
        lines.append(box_row(f"  util [{bar((util or 0) / 100)}] {util if util is not None else 0:3.0f}%   {temp}  {power}"))
        vram_txt = (f"{g['mem_used']:.1f}/{g['mem_total']:.1f} GiB"
                    if vram_frac is not None else 'n/a')
        lines.append(box_row(f"  vram [{bar(vram_frac)}] {vram_txt}"))
        combo_txt = '-'
        if sweep:
            try:
                lane_key = int(g['index'])
            except (TypeError, ValueError):
                lane_key = g['index']
            entry = sweep.get('my_lanes', {}).get(lane_key)
            if entry:
                cid, ts = entry
                age = ''
                try:
                    secs = time.time() - float(ts)
                    age = (f'  (claimed {secs / 3600:.1f}h)' if secs >= 3600
                           else f'  (claimed {secs / 60:.0f}m)')
                except (TypeError, ValueError):
                    pass
                combo_txt = f'{cid}{age}'
        lines.append(box_row(f"  combo {combo_txt}"))

    lines.append(box_sep('host'))
    if cpu is not None:
        lines.append(box_row(f"CPU    [{bar(cpu / 100)}] {cpu:3.0f}%"))
    else:
        lines.append(box_row('CPU    n/a (psutil missing)'))
    ram_pct, ram_used, ram_total, ram_source = ram
    if ram_pct is not None:
        lines.append(box_row(f"RAM    [{bar(ram_pct / 100)}] {ram_pct:3.0f}%  "
                             f"{ram_used:.1f}/{ram_total:.1f} GiB ({ram_source})"))
    else:
        lines.append(box_row('RAM    n/a'))
    if disk_rw is not None:
        lines.append(box_row(f"DISK   read {disk_rw[0]:8.1f} MB/s   write {disk_rw[1]:8.1f} MB/s"))
    else:
        lines.append(box_row('DISK   n/a'))

    lines.append(box_sep(f'sweep @ {Path(OUT_DIR).name}  (rescan every {SWEEP_SCAN_SECONDS}s, last {sweep_stamp})'))
    if sweep is None:
        lines.append(box_row('out_dir not created yet'))
    else:
        counted = sweep['done'] + sweep['failed'] + sweep['live'] + sweep['stale'] + sweep['ckpt']
        denom = GRID_TOTAL or counted
        pct = 100.0 * sweep['done'] / denom if denom else 0.0
        denom_txt = str(denom) if GRID_TOTAL else f'>={denom}'
        lines.append(box_row(f"done   [{bar(sweep['done'] / denom if denom else 0)}] "
                             f"{sweep['done']}/{denom_txt} ({pct:.1f}%)"))
        live_by = '  '.join(f'{vm}={n}' for vm, n in sorted(sweep['live_by'].items())) or '-'
        lines.extend(box_wrap(f"live   {sweep['live']:3d}   ", live_by))
        stale_by = '  '.join(f'{vm}={n}' for vm, n in sorted(sweep['stale_by'].items())) or '-'
        lines.extend(box_wrap(f"stale  {sweep['stale']:3d}   ", stale_by + '   (expired, reclaimable)'))
        lines.append(box_row(f"failed {sweep['failed']:3d}   ckpt-only {sweep['ckpt']:3d}"
                             '   (details: check_paralle.ipynb)'))
        alive = ', '.join(sweep['vms_alive']) or '-'
        lines.extend(box_wrap(f"VMs    alive {len(sweep['vms_alive'])}/{sweep['vms_total']}: ", alive))
    lines.append(box_bottom())
    return '\n'.join(lines)


def _as_pre(text):
    return ('<pre style="font-family:ui-monospace,Consolas,monospace;'
            'font-size:13px;line-height:1.2;margin:0">'
            + html.escape(text) + '</pre>')


def dashboard(panel, stop_event, interval=TICK_SECONDS):
    last_disk = psutil.disk_io_counters() if psutil else None
    last_t = time.time()
    if psutil:
        psutil.cpu_percent(interval=None)
    sweep, last_scan, sweep_stamp = None, 0.0, 'never'
    while not stop_event.is_set():
        gpus, gpu_err = get_gpu_stats()
        cpu = psutil.cpu_percent(interval=None) if psutil else None
        ram = get_memory_status()
        disk_rw = None
        if psutil:
            now_disk = psutil.disk_io_counters()
            now_t = time.time()
            dt = max(now_t - last_t, 1e-6)
            disk_rw = ((now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt,
                       (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt)
            last_disk, last_t = now_disk, now_t
        if time.time() - last_scan >= SWEEP_SCAN_SECONDS:
            try:
                sweep = scan_sweep()
                sweep_stamp = time.strftime('%H:%M:%S')
            except Exception as exc:
                sweep_stamp = f'scan error: {exc}'
            last_scan = time.time()
        panel.value = _as_pre(render(gpus, gpu_err, cpu, ram, disk_rw, sweep, sweep_stamp))
        stop_event.wait(interval)


stop_watcher('system')
system_panel = widgets.HTML(_as_pre('starting dashboard...'),
                            layout={'border': '1px solid #ddd', 'padding': '6px'})
display(system_panel)
system_stop = threading.Event()
system_thread = threading.Thread(target=dashboard, args=(system_panel, system_stop), daemon=True)
WATCHERS['system'] = {'stop': system_stop, 'thread': system_thread, 'output': system_panel}
system_thread.start()
print('dashboard watcher started in background (stop via Cell 4)')


In [ ]:
# Cell 4: stop background realtime watchers.
WATCHERS = globals().get('WATCHERS', {})
for name, item in list(WATCHERS.items()):
    item['stop'].set()
    print(f'stopped {name} watcher')
